In [1]:
import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

import requests
import pandas as pd
from datetime import date, timedelta
from functools import reduce
import os
from shutil import which
from bs4 import BeautifulSoup

chromedriver_autoinstaller.install()

'c:\\Python310\\lib\\site-packages\\chromedriver_autoinstaller\\155\\chromedriver.exe'

In [2]:
def _setup_chrome_options():
    """Set up Chrome options for local Selenium execution."""
    options = ChromeOptions()
    
    # Keep Chrome visible while running locally.
    # Uncomment this when you want headless execution again.
    # options.add_argument("--headless=new")
    
    options.add_argument("--start-maximized")
    
    return options

In [3]:
url = "https://www.rotowire.com/hockey/nhl-lineups.php"
# url = "https://www.rotowire.com/hockey/nhl-lineups.php?date=tomorrow"

options = _setup_chrome_options()
driver = webdriver.Chrome(options=options)

driver.set_page_load_timeout(20)
driver.set_script_timeout(20)

driver.get(url)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: NHL Starting Lineups: Hockey Daily Lineups | RotoWire
Current URL: https://www.rotowire.com/hockey/nhl-lineups.php


In [4]:
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "lineups"))
)

print("Lineup container found!")

Lineup container found!


In [5]:
matchups = []

lineups = (
    driver.find_element(By.CLASS_NAME, "lineups")
    .find_elements(By.CLASS_NAME, "is-nhl")
)

print("Number of lineups found:", len(lineups))

Number of lineups found: 12


In [7]:
for lineup in lineups[:-1]:
    goalies = lineup.find_elements(
        By.CLASS_NAME,
        "lineup__player-highlight-name"
    )

    away_goalie = goalies[0].text.strip().split()[-1]
    home_goalie = goalies[1].text.strip().split()[-1]

    teams = lineup.find_element(
        By.CLASS_NAME,
        "lineup__matchup"
    ).find_elements(By.TAG_NAME, "a")

    if len(teams) >= 2:
        away_team = teams[0].text.split(" (")[0]
        home_team = teams[1].text.split(" (")[0]

        matchups.append(
            (away_team, home_team, away_goalie, home_goalie)
        )

matchups

[('Islanders', 'Senators', 'Varlamov', 'Ullmark'),
 ('Rangers', 'Devils', 'Garand', 'Daws'),
 ('Canucks', 'Panthers', 'Merilainen', 'Schmid'),
 ('Sabres', 'Maple Leafs', 'Luukkonen', 'Bobrovsky'),
 ('Kraken', 'Lightning', 'Daccord', 'Vasilevskiy'),
 ('Flyers', 'Red Wings', 'Vladar', 'Tarasov'),
 ('Jets', 'Blackhawks', 'Skinner', 'Knight'),
 ('Blues', 'Wild', 'Hofer', 'Wallstedt'),
 ('Sharks', 'Predators', 'Nedeljkovic', 'Saros'),
 ('Mammoth', 'Oilers', 'Vejmelka', 'Levi'),
 ('Flames', 'Golden Knights', 'Wolf', 'Hart'),
 ('Islanders', 'Senators', 'Varlamov', 'Ullmark'),
 ('Rangers', 'Devils', 'Garand', 'Daws'),
 ('Canucks', 'Panthers', 'Merilainen', 'Schmid'),
 ('Sabres', 'Maple Leafs', 'Luukkonen', 'Bobrovsky'),
 ('Kraken', 'Lightning', 'Daccord', 'Vasilevskiy'),
 ('Flyers', 'Red Wings', 'Vladar', 'Tarasov'),
 ('Jets', 'Blackhawks', 'Skinner', 'Knight'),
 ('Blues', 'Wild', 'Hofer', 'Wallstedt'),
 ('Sharks', 'Predators', 'Nedeljkovic', 'Saros'),
 ('Mammoth', 'Oilers', 'Vejmelka', 'Levi'

In [8]:
driver.quit()

In [9]:
url = "https://moneypuck.com/goalies.htm"

options = _setup_chrome_options()
driver = webdriver.Chrome(options=options)

driver.set_page_load_timeout(20)
driver.set_script_timeout(20)

driver.get(url)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: NHL Goalie Advanced Stats 2026-2027
Current URL: https://moneypuck.com/goalies.htm


In [10]:
wait = WebDriverWait(driver, 10)

dropdown = wait.until(
    EC.presence_of_element_located((By.ID, "num_games_type"))
)

select = Select(dropdown)

select.select_by_value("_20")

print("Selected Last 20 Games")

Selected Last 20 Games


In [11]:
wait = WebDriverWait(driver, 10)

dropdown = wait.until(
    EC.presence_of_element_located((By.ID, "num_games_type"))
)

select = Select(dropdown)

select.select_by_value("_20")

print("Selected Last 20 Games")

Selected Last 20 Games


In [12]:
wait.until(
    EC.presence_of_element_located((By.ID, "goaliesTable"))
)

print("Goalie table found!")

Goalie table found!


In [13]:
table_html = driver.find_element(
    By.ID,
    "goaliesTable"
).get_attribute("outerHTML")

print("HTML length:", len(table_html))

HTML length: 154584


In [14]:
driver.quit()

In [15]:
soup = BeautifulSoup(table_html, "html.parser")

rows = soup.find_all("tr")[1:]

data = []

for tr in rows:
    tds = tr.find_all("td")

    if len(tds) < 7:
        continue

    try:
        rank = int(tds[0].get_text(strip=True))
        goalie = tds[2].get_text(strip=True).split()[-1]
        gsa = float(tds[6].get_text(strip=True))
    except:
        continue

    data.append({
        "Rank": rank,
        "Goalie": goalie,
        "GSAx": gsa
    })

goalie_df = pd.DataFrame(data)

goalie_df.head(20)

,Rank,Goalie,GSAx
0,1,Ullmark,6.9
1,2,Wallstedt,6.9
2,3,Oettinger,6.6
3,4,Sorokin,5.6
4,5,Shesterkin,4.9
5,6,Swayman,4.9
6,7,Bobrovsky,4.7
7,8,Grubauer,4.1
8,9,Tarasov,4.0
9,10,Thompson,3.4


In [16]:
goalie_ranks = dict(
    zip(goalie_df["Goalie"], goalie_df["Rank"])
)

goalie_ranks

{'Ullmark': 1,
 'Wallstedt': 2,
 'Oettinger': 3,
 'Sorokin': 4,
 'Shesterkin': 5,
 'Swayman': 6,
 'Bobrovsky': 7,
 'Grubauer': 8,
 'Tarasov': 9,
 'Thompson': 10,
 'Skinner': 11,
 'Garand': 12,
 'Woll': 13,
 'Kochetkov': 14,
 'Markstrom': 15,
 'Hofer': 16,
 'Merilinen': 17,
 'Nedeljkovic': 18,
 'Fowler': 19,
 'Husso': 20,
 'Murashov': 21,
 'Hildeby': 22,
 'Dostal': 23,
 'Daccord': 24,
 'Vejmelka': 25,
 'Murray': 26,
 'Varlamov': 27,
 'DeSmith': 28,
 'Hart': 29,
 'Silovs': 30,
 'Lindgren': 31,
 'Ellis': 32,
 'Saros': 33,
 'Vasilevskiy': 34,
 'Hill': 35,
 'Wedgewood': 36,
 'Levi': 37,
 'Cossa': 38,
 'DiPietro': 39,
 'Knight': 40,
 'Stevenson': 41,
 'Stolarz': 42,
 'Allen': 43,
 'Greaves': 44,
 'Schmid': 45,
 'Askarov': 46,
 'Talbot': 47,
 'Vladar': 48,
 'Pickard': 49,
 'Bussi': 50,
 'Forsberg': 51,
 'Soderblom': 52,
 'Gibson': 53,
 'Daws': 54,
 'Jarry': 55,
 'Blackwood': 56,
 'Luukkonen': 57,
 'Binnington': 58,
 'Ersson': 59,
 'Cooley': 60,
 'Kuemper': 61,
 'Dobes': 62,
 'Wolf': 63,
 'Lan

In [17]:
url = "https://www.nextteamup.com/nhl"

headers = {
    "User-Agent": "Mozilla/5.0"
}

resp = requests.get(
    url,
    headers=headers,
    timeout=15
)

resp.raise_for_status()

print("Status:", resp.status_code)
print("Response length:", len(resp.text))

Status: 200
Response length: 59315


In [18]:
soup = BeautifulSoup(resp.text, "html.parser")

streaks = {}

rows = soup.select("table tbody tr")

print("Rows found:", len(rows))

Rows found: 32


In [19]:
# Grab all table rows (skip header automatically)
rows = soup.select("table tbody tr")

for row in rows:
    tds = row.find_all("td")
    if len(tds) < 3:
        continue

    team = tds[1].get_text(strip=True)
    streak = tds[2].get_text(strip=True)

    # Convert "+7" / "-3" → int
    try:
        streak = int(streak)
    except ValueError:
        continue

    streaks[team] = streak
    
streaks

{'Washington Capitals': 4,
 'St. Louis Blues': 4,
 'Florida Panthers': 3,
 'Carolina Hurricanes': 3,
 'Chicago Blackhawks': 1,
 'New York Rangers': 1,
 'Calgary Flames': 1,
 'San Jose Sharks': 1,
 'Buffalo Sabres': -1,
 'Tampa Bay Lightning': -1,
 'Boston Bruins': -1,
 'Edmonton Oilers': -1,
 'Pittsburgh Penguins': -1,
 'Vancouver Canucks': -1,
 'New Jersey Devils': -1,
 'Anaheim Ducks': -2,
 'Minnesota Wild': -2,
 'Nashville Predators': -2,
 'Columbus Blue Jackets': -2,
 'Vegas Golden Knights': -3,
 'Utah Mammoth': -3,
 'Dallas Stars': -3,
 'Seattle Kraken': -3,
 'Detroit Red Wings': -3,
 'New York Islanders': -3,
 'Montreal Canadiens': -4,
 'Colorado Avalanche': -4,
 'Philadelphia Flyers': -4,
 'Ottawa Senators': -4,
 'Winnipeg Jets': -4,
 'Los Angeles Kings': -6,
 'Toronto Maple Leafs': -7}

In [28]:
import os
from dotenv import load_dotenv

load_dotenv()

key = os.getenv("NST_API_KEY")

if not key:
    raise ValueError("NST_API_KEY not found. Check your .env file.")

In [29]:
from io import StringIO

# Date range
today = date.today()
start = today - timedelta(days=30)

# Your NST access key
key = "451fe70942969369f896204690d6cd68"

# Build the URL using NST's data subdomain
url = (
    "https://data.naturalstattrick.com/teamtable.php"
    f"?key={key}"
    f"&fromseason=20252026"
    f"&thruseason=20252026"
    f"&stype=2"
    f"&sit=5v5"
    f"&score=all"
    f"&rate=n"
    f"&team=all"
    f"&loc=B"
    f"&gpf=410"
    f"&fd={start}"
    f"&td={today}"
)

print("Requesting NST team table...")

response = requests.get(
    url,
    headers={"nst-key": key},
    timeout=30
)
response.raise_for_status()

html = response.text

print("Status code:", response.status_code)
print("HTML length:", len(html))
print("Contains table:", "<table" in html.lower())

# Parse the HTML table into a DataFrame
df = pd.read_html(
    StringIO(html),
    flavor="lxml",
    header=0,
    index_col=0,
    na_values=["-"]
)[0]

print("DataFrame shape:", df.shape)
df.head()


Requesting NST team table...
Status code: 200
HTML length: 63305
Contains table: True
DataFrame shape: (32, 71)


,Team,GP,TOI,W,L,OTL,ROW,Points,Point %,CF,...,LDSA,LDSF%,LDGF,LDGA,LDGF%,LDSH%,LDSV%,SH%,SV%,PDO
1,Chicago Blackhawks,82,4006:50,29,39,14,26,72,0.439,3346,...,792,46.49,16,27,37.21,2.33,96.59,9.56,90.00,0.996
2,Florida Panthers,82,3877:00,40,38,4,35,84,0.512,3755,...,714,51.26,22,29,43.14,2.93,95.94,9.62,89.30,0.989
3,Pittsburgh Penguins,82,4010:59,41,25,16,38,98,0.598,3876,...,721,48.90,33,31,51.56,4.78,95.70,11.13,90.10,1.012
4,New York Rangers,82,4035:07,34,39,9,31,77,0.470,3679,...,816,45.74,26,28,48.15,3.78,96.57,9.56,91.33,1.009
5,Colorado Avalanche,82,3938:26,55,16,11,51,121,0.738,4612,...,744,55.98,47,27,63.51,4.97,96.37,9.84,92.20,1.020


In [21]:
cf = df.sort_values(by="CF%", ascending=False, ignore_index=True)
cf.index += 1
cf.reset_index(inplace=True)
cf = cf.rename(columns={"index": "CF%"})
cf = cf[["Team", "CF%"]]

gf = df.sort_values(by="GF%", ascending=False, ignore_index=True)
gf.index += 1
gf.reset_index(inplace=True)
gf = gf.rename(columns={"index": "GF%"})
gf = gf[["Team", "GF%"]]

xgf = df.sort_values(by="xGF%", ascending=False, ignore_index=True)
xgf.index += 1
xgf.reset_index(inplace=True)
xgf = xgf.rename(columns={"index": "xGF%"})
xgf = xgf[["Team", "xGF%"]]

hdcf = df.sort_values(by="HDCF%", ascending=False, ignore_index=True)
hdcf.index += 1
hdcf.reset_index(inplace=True)
hdcf = hdcf.rename(columns={"index": "HDCF%"})
hdcf = hdcf[["Team", "HDCF%"]]

sh = df.sort_values(by="SH%", ascending=False, ignore_index=True)
sh.index += 1
sh.reset_index(inplace=True)
sh = sh.rename(columns={"index": "SH%"})
sh = sh[["Team", "SH%"]]

In [22]:
cf.head()

,Team,CF%,CF%
0,Carolina Hurricanes,1,59.78
1,Colorado Avalanche,2,56.69
2,Tampa Bay Lightning,3,53.30
3,Utah Mammoth,4,53.13
4,Ottawa Senators,5,52.85


In [23]:
dfs = [cf, gf, xgf, hdcf, sh]

final_df = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on="Team",
        how="outer"
    ),
    dfs
)

final_df

,Team,CF%,CF%,GF%,GF%,xGF%,xGF%,HDCF%,HDCF%,SH%,SH%
0,Carolina Hurricanes,1,59.78,9,53.49,2,55.89,4,55.15,24,8.89
1,Colorado Avalanche,2,56.69,1,62.79,1,56.39,3,55.37,13,9.84
2,Tampa Bay Lightning,3,53.30,2,56.20,4,54.19,2,55.61,3,11.09
3,Utah Mammoth,4,53.13,10,53.16,6,52.66,8,52.75,7,10.32
4,Ottawa Senators,5,52.85,11,52.34,3,54.54,5,54.55,12,9.96
5,Anaheim Ducks,6,52.35,23,47.37,13,50.77,15,50.60,25,8.88
6,Vegas Golden Knights,7,52.31,16,50.29,5,54.02,1,56.05,18,9.35
7,Los Angeles Kings,8,52.22,12,52.05,7,52.19,6,53.16,27,8.41
8,Florida Panthers,9,51.72,24,47.23,16,50.21,18,50.33,15,9.62
9,New Jersey Devils,10,51.38,31,42.63,18,49.87,20,49.01,32,7.22


In [24]:
def normalize_team(name):
    return (
        name.lower()
        .replace(".", "")
        .replace("é", "e")
        .replace(" ", "")
    )

In [25]:
normalized_streaks = {
    normalize_team(team): streak
    for team, streak in streaks.items()
}

normalized_streaks

{'washingtoncapitals': 4,
 'stlouisblues': 4,
 'floridapanthers': 3,
 'carolinahurricanes': 3,
 'chicagoblackhawks': 1,
 'newyorkrangers': 1,
 'calgaryflames': 1,
 'sanjosesharks': 1,
 'buffalosabres': -1,
 'tampabaylightning': -1,
 'bostonbruins': -1,
 'edmontonoilers': -1,
 'pittsburghpenguins': -1,
 'vancouvercanucks': -1,
 'newjerseydevils': -1,
 'anaheimducks': -2,
 'minnesotawild': -2,
 'nashvillepredators': -2,
 'columbusbluejackets': -2,
 'vegasgoldenknights': -3,
 'utahmammoth': -3,
 'dallasstars': -3,
 'seattlekraken': -3,
 'detroitredwings': -3,
 'newyorkislanders': -3,
 'montrealcanadiens': -4,
 'coloradoavalanche': -4,
 'philadelphiaflyers': -4,
 'ottawasenators': -4,
 'winnipegjets': -4,
 'losangeleskings': -6,
 'torontomapleleafs': -7}

In [26]:
res = pd.DataFrame()

for away, home, away_goalie, home_goalie in matchups:

    away_df = final_df[
        final_df["Team"].str.contains(
            away,
            case=False,
            na=False
        )
    ]

    home_df = final_df[
        final_df["Team"].str.contains(
            home,
            case=False,
            na=False
        )
    ]

    away_rank = goalie_ranks.get(away_goalie)
    home_rank = goalie_ranks.get(home_goalie)

    away_df = away_df.assign(
        Goalie=away_goalie,
        Goalie_Rank=away_rank,
        Streak=away_df["Team"].apply(
            lambda t: normalized_streaks.get(
                normalize_team(t)
            )
        )
    )

    home_df = home_df.assign(
        Goalie=home_goalie,
        Goalie_Rank=home_rank,
        Streak=home_df["Team"].apply(
            lambda t: normalized_streaks.get(
                normalize_team(t)
            )
        )
    )

    matchup_df = pd.concat(
        [away_df, home_df],
        ignore_index=True
    )

    res = pd.concat(
        [res, matchup_df],
        ignore_index=True
    )

In [27]:
output_file = "../public/result.csv"

res.to_csv(output_file, index=False)

print(f"Saved {len(res)} rows to {output_file}")

Saved 44 rows to ../public/result.csv
